In [1]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys

MODELS_DIR = Path("/kaggle/working/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DOWNLOADS = [
    ("mistralai/Mistral-7B-Instruct-v0.3", "mistral_7b_instruct_v03"),
]

# Resolve Hugging Face token
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    print("CRITICAL: HF_TOKEN not provided. Gated/repo-specific downloads will fail.")
    sys.exit(1)

# Explicit allow list: sharded weights + HF index + configs + tokenizer assets
ALLOW_PATTERNS = [
    "config.json",
    "params.json",
    "generation_config.json",
    "model.safetensors.index.json",
    "model-*.safetensors",
    "tokenizer_config.json",
    "tokenizer.json",
    "special_tokens_map.json",
    "tokenizer.model",
    "tokenizer.model.v3",
]

REQUIRED_FILES = [
    "config.json",
    "params.json",
    "generation_config.json",
    "model.safetensors.index.json",
    "tokenizer_config.json",
    "tokenizer.json",
    "special_tokens_map.json",
    "tokenizer.model",
    "tokenizer.model.v3",
]

for repo_id, local_name in MODEL_DOWNLOADS:
    local_dir = MODELS_DIR / local_name
    
    # Check completeness before initiating download
    if local_dir.exists() and all((local_dir / f).exists() for f in REQUIRED_FILES):
        safetensor_count = len(list(local_dir.glob("model-*.safetensors")))
        if safetensor_count > 0:
            print(f"✓ {local_name} verified complete — {safetensor_count} shards. Skipping.")
            continue
            
    print(f"\n▶ Initiating download: {repo_id}")
    print(f"  Target: {local_dir}")
    print(f"  Filter: allow_patterns={ALLOW_PATTERNS}")
    
    try:
        snapshot_download(
            repo_id=repo_id,
            local_dir=str(local_dir),
            local_dir_use_symlinks=False,
            resume_download=True,
            token=HF_TOKEN,
            allow_patterns=ALLOW_PATTERNS,
            max_workers=4,
        )
    except Exception as e:
        print(f"✗ Download failed: {e}")
        sys.exit(1)
        
    # Post-download validation
    missing = [f for f in REQUIRED_FILES if not (local_dir / f).exists()]
    safetensors = list(local_dir.glob("model-*.safetensors"))
    
    if missing:
        print(f"✗ Missing critical assets: {missing}")
        sys.exit(1)
    if not safetensors:
        print("✗ No model weight shards downloaded.")
        sys.exit(1)
        
    total_weights_gb = sum(f.stat().st_size for f in safetensors) / (1024**3)
    print(f"✓ {local_name} validated — {len(safetensors)} shards ({total_weights_gb:.2f} GB)")


▶ Initiating download: mistralai/Mistral-7B-Instruct-v0.3
  Target: /kaggle/working/models/mistral_7b_instruct_v03
  Filter: allow_patterns=['config.json', 'params.json', 'generation_config.json', 'model.safetensors.index.json', 'model-*.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'special_tokens_map.json', 'tokenizer.model', 'tokenizer.model.v3']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

✓ mistral_7b_instruct_v03 validated — 3 shards (13.50 GB)


In [2]:
import os
import subprocess, sys

print("▶ Disk usage summary:")
result = subprocess.run(["du", "-sh", str(MODELS_DIR)],
                        capture_output=True, text=True)
print(result.stdout)

print("\n▶ Model directory contents:")
for model_dir in sorted(MODELS_DIR.iterdir()):
    files = list(model_dir.iterdir())
    size_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1024**2
    print(f"  {model_dir.name}/  ({len(files)} files, {size_mb:.0f} MB)")
    for f in sorted(files)[:5]:
        print(f"    {f.name}")
    if len(files) > 5:
        print(f"    ... ({len(files)} total)")

print("\n✓ Setup complete. Commit this notebook to save outputs as Kaggle dataset.")

▶ Disk usage summary:
14G	/kaggle/working/models


▶ Model directory contents:
  mistral_7b_instruct_v03/  (13 files, 13828 MB)
    .cache
    config.json
    generation_config.json
    model-00001-of-00003.safetensors
    model-00002-of-00003.safetensors
    ... (13 total)

✓ Setup complete. Commit this notebook to save outputs as Kaggle dataset.
